# Name That LLM: offline submission (parent blend + native expert + L24 expert)

Predicts the model behind each test passage and writes `submission.parquet`. Everything runs with **internet off**. Built on the 0.657 notebook (`ntl-submission-native` v2).

**Parent blend** (stage-2 Gemma 3 4B LoRA x2, n-gram SVC, Qwen3-Embedding and BERT logistic regressions, prompt-centred with groups estimated from the text; public score 0.551 alone).

**Native expert** (CHEN Han's published Qwen3-Embedding-8B classifier, five prompt-fold adapters; dataset `lightningv08/ntl-olmo2-native24-final-inference-assets`). Inference only, my own port in `native_worker.py`.

**New: L24 expert.** The public Qwen3-8B-Base classifier from `dhruvtibarewal/nlm-l24-adapters-v5` (`q8v5`): the base model cut to its first 24 layers, a rank-16 LoRA, eight pooled depths with a LayerNorm each, and plain and prompt-centred heads. Inference only, my own port in `l24_worker.py`; all five fold adapters are scored on every test passage and averaged, using the centred head inside estimated groups of 2-18 passages and the plain head elsewhere.

**Why this blend.** On the 2,699 official rows, scored out of fold (each row by an adapter that never saw its prompt): native expert 0.681, L24 expert (centred head, estimated groups) 0.665, and the two together at 50/50 0.704, a gain of +0.023 (95% interval +0.013 to +0.033) over the native expert alone; they agree on only 68.5% of passages. On the 539 rows stage 2 held out, adding the L24 expert to the previous recipe moved 0.703 to 0.717-0.721 (interval includes zero at that size). Weights: parent 0.30, native 0.35, L24 0.35, each divided by its standard deviation first.

**Inputs to add**: the competition data; notebook outputs `notebook6fafdc1c20`, `ntl-stage-frozen-models`, `ntl-stage2-official`; datasets `lightningv08/ntl-olmo2-native24-final-inference-assets` and `dhruvtibarewal/nlm-l24-adapters-v5`; models `qwen-lm/qwen-3-embedding` (transformers, 8b) and `qwen-lm/qwen-3` (transformers, 8b-base). GPU T4 x2, internet off.

**Running it.** The commit run uses a 3-row stand-in test, so group estimation is skipped there. Measured timings on 2,699 passages: native expert 97 s per fold per 540 passages; L24 expert 4.7 min per adapter, so about 7 min per adapter and 35 min in all for 4,000 passages. The notebook stops with an error rather than quietly submitting a weaker blend (`REQUIRE_NATIVE`, `REQUIRE_L24`). The two-way (parent + native) predictions are kept in `submission_two_way.parquet`.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path

VALIDATE = False       # off: the stage-2 adapter has seen every official row, so held-out scores would be meaningless
VAL_PROMPTS = 40
VARIANT = 'recipe'     # 'recipe' (default), 'gemma_only' or 'no_centering': the last two exist only to probe the real test score
NATIVE_WEIGHT = 0.65    # share of the final z-scored blend given to the native expert (0.6-0.8 scored about the same on held-out rows)
W_PARENT, W_NATIVE, W_L24 = 0.30, 0.35, 0.35   # final z-scored blend weights
REQUIRE_L24 = True     # stop with an error rather than submit without the L24 expert
REQUIRE_NATIVE = True  # stop with an error rather than submit the parent blend alone
REQUIRE_GEMMA = True   # stop with an error rather than submit without the Gemma scores
USE_ASSIGNMENT = False # assignment scored 0.517 against 0.540 without it on the real test, so it stays off
NGRAM_OFFICIAL_WEIGHT = 1.0   # official rows join the n-gram training (+0.017 on held-out prompts in a local test)
def find(pattern, what):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    if not hits: raise SystemExit(f'Could not find {what} (looked for {pattern}). Inputs present: {sorted(os.listdir("/kaggle/input"))}')
    return hits[0]

COMP = next((p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv')), None)
assert COMP, 'Add the Name That LLM competition data as an input.'
BASE = str(Path(find('offline/base_gemma/config.json', 'the Gemma base model copy')).parent)
WHEELS = find('offline/wheels', 'the saved packages')
ADAPTER = str(Path(find('stage2/runs/final/adapter/adapter_model.safetensors', 'the stage-2 LoRA adapter (add the ntl-stage2-official output)')).parent)
HEAD = find('stage2/runs/final/head.pt', 'the stage-2 classifier head')
SAVED = find('stage2/runs/final/pred.npz', 'the stage-2 scores on 64 official rows')
POOL = find('run/train_pool.parquet', 'the pool training rows')
QWEN = str(Path(find('frozen/qwen_emb/config.json', 'the Qwen3-Embedding weights (run the staging notebook)')).parent)
BERT = str(Path(find('frozen/bert/config.json', 'the BERT weights (run the staging notebook)')).parent)
def first(paths, what):
    for p in paths:
        if os.path.exists(p): return p
    raise SystemExit(f'Could not find {what}. Tried {paths}. /kaggle/input has: {sorted(os.listdir("/kaggle/input"))}')
ASSETS = first(['/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679',
                '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679'], 'the native expert assets (add the dataset lightningv08/ntl-olmo2-native24-final-inference-assets)')
NATIVE_BASE = first(['/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1', '/kaggle/input/qwen-3-embedding/transformers/8b/1'], 'the Qwen3-Embedding-8B model (add qwen-lm/qwen-3-embedding, transformers, 8b)')
L24_ADAPTERS = first(['/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5', '/kaggle/input/nlm-l24-adapters-v5'], 'the L24 adapters (add the dataset dhruvtibarewal/nlm-l24-adapters-v5)')
Q8_BASE = first(['/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1', '/kaggle/input/qwen-3/transformers/8b-base/1'], 'the Qwen3-8B-Base model (add qwen-lm/qwen-3, transformers, 8b-base)')
for name, value in dict(L24_ADAPTERS=L24_ADAPTERS, Q8_BASE=Q8_BASE, ASSETS=ASSETS, NATIVE_BASE=NATIVE_BASE, COMP=COMP, BASE=BASE, WHEELS=WHEELS, ADAPTER=ADAPTER, HEAD=HEAD, SAVED=SAVED, POOL=POOL, QWEN=QWEN, BERT=BERT).items(): print(f'{name:8s} {value}')


In [ ]:
%%writefile score_gemma.py
"""Score passages with the trained Gemma LoRA classifier, offline. Mirrors the scoring path of worker_unsloth.py step for step:
same model load, same LoRA layout, same decoder, same three pooled layers, same head. Writes log-probabilities to --out.

  python score_gemma.py --check                                   # only import the stack and print versions
  python score_gemma.py --base DIR --adapter DIR --head head.pt --texts texts.parquet --out out.npz
"""
import argparse, importlib.metadata, importlib.util, sys, time
from pathlib import Path


def _torchao_too_old():
    try:
        version = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        return False
    from packaging.version import parse
    return parse(version) < parse('0.16.0')


if _torchao_too_old():   # peft refuses old torchao builds. Nothing here uses torchao, so hide it before peft looks.
    _find_spec = importlib.util.find_spec
    importlib.util.find_spec = lambda name, *args, **kw: None if name.split('.')[0] == 'torchao' else _find_spec(name, *args, **kw)

ap = argparse.ArgumentParser()
ap.add_argument('--check', action='store_true')
ap.add_argument('--base'); ap.add_argument('--adapter'); ap.add_argument('--head'); ap.add_argument('--texts'); ap.add_argument('--out')
ap.add_argument('--max-len', type=int, default=384); ap.add_argument('--seed', type=int, default=1); ap.add_argument('--rank', type=int, default=16)
a = ap.parse_args()

import unsloth  # noqa: F401  (must load before transformers and peft)
from unsloth import FastModel
import numpy as np, pandas as pd, torch
from torch import nn

if a.check:
    import bitsandbytes, peft, transformers
    print('unsloth', unsloth.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__,
          '| bitsandbytes', bitsandbytes.__version__, '| gpus', torch.cuda.device_count())
    sys.exit(0)

T0 = time.time()
def log(*x): print(f'[{(time.time() - T0) / 60:6.1f}m]', *x, flush=True)

dev = 'cuda'; compute = torch.float16    # the training run used the default, fp16 (Unsloth keeps Gemma's activations in fp32 itself)
texts = pd.read_parquet(a.texts).text.fillna('').astype(str).tolist()

model, processor = FastModel.from_pretrained(model_name=a.base, max_seq_length=a.max_len, dtype=compute, load_in_4bit=True,
                                             full_finetuning=False, use_gradient_checkpointing='unsloth', random_state=a.seed)
tok = getattr(processor, 'tokenizer', processor)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'right'
model = FastModel.get_peft_model(model, r=a.rank, lora_alpha=2 * a.rank, lora_dropout=0, bias='none', random_state=a.seed,
                                 finetune_vision_layers=False, finetune_language_layers=True,
                                 finetune_attention_modules=True, finetune_mlp_modules=True, use_gradient_checkpointing='unsloth')
log('loaded base model and LoRA layout')


def find_decoder(m):
    best = None
    for name, mod in m.named_modules():
        if any(w in name.lower() for w in ('vision', 'audio', 'visual')): continue
        layers = getattr(mod, 'layers', None)
        if isinstance(layers, nn.ModuleList) and hasattr(mod, 'embed_tokens') and (best is None or len(layers) > len(best[1].layers)):
            best = (name, mod)
    if best is None: raise SystemExit('could not find the text decoder inside the model')
    return best
dec_name, decoder = find_decoder(model)
n_layers = len(decoder.layers); picks = sorted({n_layers // 2, 3 * n_layers // 4, n_layers})
hidden = int(getattr(getattr(model.config, 'text_config', model.config), 'hidden_size'))
log('decoder', dec_name or '(root)', n_layers, 'layers, pooling', picks)


class Head(nn.Module):
    def __init__(self):
        super().__init__(); self.norms = nn.ModuleList(nn.LayerNorm(hidden) for _ in picks); self.out = nn.Linear(hidden * len(picks), 18)
    def features(self, pooled): return torch.cat([n(p) for n, p in zip(self.norms, pooled)], -1)
    def forward(self, pooled): return self.out(self.features(pooled))
head = Head().to(dev)

# same order as training: for_training first, then adapters to fp32, so the saved fp32 weights load without rounding
FastModel.for_training(model)
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32: p.data = p.data.float()

import json
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
cfg = json.loads((Path(a.adapter) / 'adapter_config.json').read_text())
if int(cfg.get('r', a.rank)) != a.rank: raise SystemExit(f'adapter rank {cfg.get("r")} differs from --rank {a.rank}')
state = load_file(str(Path(a.adapter) / 'adapter_model.safetensors'))
result = set_peft_model_state_dict(model, state)
unexpected = list(getattr(result, 'unexpected_keys', None) or [])
if unexpected: raise SystemExit(f'{len(unexpected)} adapter tensors did not match the model, first: {unexpected[:3]}')
lora_b = sum(float(p.detach().float().abs().sum()) for n, p in model.named_parameters() if 'lora_B' in n)
if not lora_b > 0: raise SystemExit('every LoRA B matrix is still zero after loading: the adapter was not applied')
head.load_state_dict(torch.load(a.head, map_location='cpu'))
log(f'adapter loaded ({len(state)} tensors, |B| sum {lora_b:.1f}) and head loaded')

enc = tok(texts, truncation=True, max_length=a.max_len)['input_ids']
def batch(rows): return tok.pad({'input_ids': [enc[int(i)] for i in rows]}, return_tensors='pt').to(dev)
def pool_layers(b):
    with torch.autocast(dev, dtype=compute):
        hs = decoder(input_ids=b['input_ids'], attention_mask=b['attention_mask'], output_hidden_states=True, use_cache=False).hidden_states
    if hs is None or len(hs) != n_layers + 1: raise SystemExit(f'the decoder returned {None if hs is None else len(hs)} hidden states, expected {n_layers + 1}')
    m = b['attention_mask'].clone().float(); m[:, 0] = 0; m = m.unsqueeze(-1); den = m.sum(1).clamp(min=1)
    return [(hs[l].float() * m).sum(1) / den for l in picks]

model.eval(); head.eval()
N = len(enc); logp = np.zeros((N, 18), np.float32)
with torch.inference_mode():
    idx = np.argsort([len(x) for x in enc], kind='stable')
    for s in range(0, N, 16):
        rows = idx[s:s + 16]
        logp[rows] = torch.log_softmax(head.out(head.features(pool_layers(batch(rows)))), -1).cpu().numpy()
        if (s // 16) % 50 == 0: log(f'scored {s}/{N}')
if not np.isfinite(logp).all(): raise SystemExit('non-finite predictions')
np.savez(a.out, logp=logp)
log(f'done: {N} passages scored, saved to {a.out}')


In [ ]:
%%writefile ntl_pipeline.py
"""Name That LLM: offline submission pipeline.

Recipe (validated in experiments/sim_unequal.py): 2 x Gemma LoRA scores (raw) + n-gram scores (centred per prompt group)
+ Qwen3-Embedding mid-layer LR (centred) + BERT mid-layer LR (centred). "Centred" means each prompt group's mean is subtracted.
The test set has no prompt ids, so groups are estimated from the text: Ward clustering of Qwen last-layer features, with the
number of clusters chosen by silhouette (on training rows it lands within 0.91-1.08x of the true prompt count).

run_pipeline() takes the Gemma scorer as a function so everything else can be tested without Unsloth.
"""
import json, time
from pathlib import Path
import numpy as np, pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.optimize import linear_sum_assignment
from scipy.special import log_softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.svm import LinearSVC

WEIGHTS = {'gemma': 2.0, 'ngram': 1.0, 'qwen': 1.0, 'bert': 1.0}


def macro_recall(y, p):
    return float(np.mean([(p[y == c] == c).mean() for c in np.unique(y)]))


def center_by_group(X, groups):
    X = np.asarray(X, dtype=np.float64).copy()
    for g in np.unique(groups):
        m = groups == g
        X[m] -= X[m].mean(0, keepdims=True)
    return X


def assign_by_group(S, groups):
    """One model per passage inside each group, each model used at most ceil(size/18) times. Only valid if prompts really
    hold one passage per model, which the competition says is not guaranteed. Off by default."""
    pred = S.argmax(1).copy(); k = S.shape[1]
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        r, c = linear_sum_assignment(-np.tile(S[idx], (1, int(np.ceil(len(idx) / k)))))
        pred[idx[r]] = c % k
    return pred


def estimate_groups(X, lo=40, hi=4, n_grid=24):
    """Cluster passages into prompt groups from text features alone. Returns (labels, info)."""
    N = len(X)
    if N < 30:
        return np.zeros(N, dtype=int), {'K': 1, 'note': f'only {N} rows: no grouping'}
    Xn = normalize(X - X.mean(0))
    Z = linkage(Xn, 'ward'); D = pairwise_distances(Xn)
    best = None
    for K in sorted({int(round(k)) for k in np.geomspace(max(2, N / lo), max(3, N / hi), n_grid)}):
        lab = fcluster(Z, K, 'maxclust')
        if not 1 < len(np.unique(lab)) < N: continue
        s = silhouette_score(D, lab, metric='precomputed')
        if best is None or s > best[1]: best = (K, s, lab)
    if best is None:
        return np.zeros(N, dtype=int), {'K': 1, 'note': 'no usable clustering'}
    K, s, lab = best
    lab = np.unique(lab, return_inverse=True)[1]; sizes = np.bincount(lab)
    return lab, {'K': int(len(sizes)), 'silhouette': float(s), 'size_min': int(sizes.min()), 'size_median': float(np.median(sizes)),
                 'size_mean': float(sizes.mean()), 'size_max': int(sizes.max()), 'singletons': int((sizes == 1).sum())}


def ngram_scores(pool, labels, text_sets, log=print, extra=None, extra_weight=1.0):
    """The same n-gram SVC the training notebook scored the official rows with, trained on the pool rows (plus, optionally, official rows
    at extra_weight: extra = (texts, labels))."""
    t = time.time()
    svc = Pipeline([('f', FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=300000, sublinear_tf=True, lowercase=False, dtype=np.float32)),
        ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=100000, sublinear_tf=True, lowercase=False, token_pattern=r'\w+|[^\w\s]', dtype=np.float32))])),
        ('clf', LinearSVC(C=0.5, class_weight='balanced', max_iter=5000))])
    X, Y = pool.text.tolist(), pool.model.tolist(); sw = np.ones(len(X))
    if extra is not None:
        X += list(extra[0]); Y += list(extra[1]); sw = np.r_[sw, np.full(len(extra[0]), float(extra_weight))]
    svc.fit(X, Y, clf__sample_weight=sw)
    cols = [list(svc.classes_).index(l) for l in labels]
    log(f'n-gram fitted on {len(X)} rows ({len(pool)} pool + {len(X) - len(pool)} official) in {(time.time() - t) / 60:.1f} min')
    return [log_softmax(svc.decision_function(texts)[:, cols] * 4, axis=1) if len(texts) else np.zeros((0, len(labels))) for texts in text_sets]


def embed(path, texts, depths, log=print, batch=16, max_len=512):
    """Mean-pooled hidden states. depths are fractions of the layer count (0.5 = middle, 1.0 = last). Returns {layer: array}."""
    import torch
    from transformers import AutoModel, AutoTokenizer
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModel.from_pretrained(path).float().to(dev).eval()
    n = model.config.num_hidden_layers; layers = sorted({max(1, round(f * n)) for f in depths})
    texts = list(texts); order = np.argsort([len(t) for t in texts]); out = {L: np.zeros((len(texts), model.config.hidden_size), np.float32) for L in layers}
    t = time.time()
    with torch.no_grad():
        for i in range(0, len(order), batch):
            idx = order[i:i + batch]
            enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(dev)
            hs = model(**enc, output_hidden_states=True).hidden_states
            m = enc['attention_mask'].unsqueeze(-1).float()
            for L in layers: out[L][idx] = ((hs[L].float() * m).sum(1) / m.sum(1)).cpu().numpy()
    del model
    if dev == 'cuda': torch.cuda.empty_cache()
    log(f'embedded {len(texts)} passages with {Path(path).name} ({n} layers, pooled {layers}) in {(time.time() - t) / 60:.1f} min')
    return out


def frozen_scores(E_train, y_train, g_train, E_apply, g_apply, C=0.01, center=True):
    """Logistic regression on prompt-centred features. Training rows are centred by their true prompt, rows to score by their estimated group."""
    Xtr = center_by_group(E_train, g_train) if center else np.asarray(E_train, dtype=np.float64); sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(C=C, max_iter=400).fit(sc.transform(Xtr), y_train)
    return clf.decision_function(sc.transform(center_by_group(E_apply, g_apply) if center else np.asarray(E_apply, dtype=np.float64)))


def blend(parts, weights=WEIGHTS):
    return sum(weights[k] * (v / v.std()) for k, v in parts.items())


def run_pipeline(comp_dir, pool_path, qwen_dir, bert_dir, gemma_fn, out_dir, validate=True, val_prompts=40, require_gemma=True,
                 use_assignment=False, holdout_prompts=(), test_df=None, pool_limit=None, variant='recipe', ngram_official_weight=0.0, log=print):
    """variant: 'recipe' (default), 'gemma_only' (Gemma scores alone, to see what it really scores on the test) or 'no_centering'."""
    t0 = time.time(); out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(f'{comp_dir}/train.csv'); train['text'] = train.text.fillna('').astype(str)
    test = test_df.copy() if test_df is not None else pd.read_csv(f'{comp_dir}/test.csv'); test['text'] = test.text.fillna('').astype(str)
    labels = sorted(train.model.unique()); y = train.model.map({l: i for i, l in enumerate(labels)}).to_numpy(); g_true = train.prompt_id.to_numpy()
    usable = ~np.isin(g_true, list(holdout_prompts))   # rows allowed to train the frozen models (everything, except in local tests)
    rng = np.random.default_rng(0)
    val_set = set(rng.choice(np.unique(g_true[usable]), val_prompts, replace=False)) if validate else set()
    val_idx = np.where(np.isin(g_true, list(val_set)) & usable)[0]
    log(f'{len(train)} train rows, {len(test)} test rows; validation on {len(val_idx)} held-out train rows ({len(val_set)} prompts)')

    # 1. Gemma LoRA scores for the validation rows and the test rows in one call
    texts = list(train.text.iloc[val_idx]) + list(test.text)
    try:
        G = np.asarray(gemma_fn(texts), dtype=np.float64); gemma_ok = True
    except Exception as error:
        if require_gemma: raise
        log(f'!!! GEMMA FAILED, continuing without it: {error!r}'); G = np.zeros((len(texts), len(labels))); gemma_ok = False
    G_val, G_test = G[:len(val_idx)], G[len(val_idx):]

    # 2. n-gram scores from the pool rows
    pool = pd.read_parquet(pool_path, columns=['text', 'model'])
    if pool_limit: pool = pool.sample(min(pool_limit, len(pool)), random_state=0)
    extra = None
    if ngram_official_weight > 0:   # official rows join the n-gram training, except validation rows so validation stays honest
        keep = usable.copy(); keep[val_idx] = False
        extra = (list(train.text[keep]), list(train.model[keep]))
    N_val, N_test = ngram_scores(pool, labels, [train.text.iloc[val_idx], test.text], log, extra=extra, extra_weight=ngram_official_weight)

    # 3. frozen embedders on all training rows and on the test rows
    q_tr = embed(qwen_dir, train.text, (0.5, 1.0), log); q_te = embed(qwen_dir, test.text, (0.5, 1.0), log)
    b_tr = embed(bert_dir, train.text, (0.5,), log); b_te = embed(bert_dir, test.text, (0.5,), log)
    qmid, qlast = sorted(q_tr)[0], sorted(q_tr)[-1]; bmid = sorted(b_tr)[0]

    def recipe(G_, N_, qs_tr_mask, q_apply, b_apply, last_apply, true_labels=None):
        est, info = estimate_groups(last_apply)
        if variant == 'gemma_only':
            parts = {'gemma': G_}
        else:
            c = variant != 'no_centering'
            fq = frozen_scores(q_tr[qmid][qs_tr_mask], y[qs_tr_mask], g_true[qs_tr_mask], q_apply, est, center=c)
            fb = frozen_scores(b_tr[bmid][qs_tr_mask], y[qs_tr_mask], g_true[qs_tr_mask], b_apply, est, center=c)
            parts = {'gemma': G_, 'ngram': center_by_group(N_, est) if c else N_, 'qwen': fq, 'bert': fb}
        if not gemma_ok: parts.pop('gemma', None)
        S = blend(parts, {k: WEIGHTS[k] for k in parts})
        pred = assign_by_group(S, est) if use_assignment else S.argmax(1)
        return S, pred, parts, est, info

    report = {'weights': WEIGHTS, 'gemma_used': gemma_ok, 'use_assignment': bool(use_assignment), 'variant': variant, 'ngram_official_weight': ngram_official_weight}
    # 4. validation: frozen models see only training prompts outside the validation prompts
    if validate and len(val_idx):
        mask = usable.copy(); mask[val_idx] = False
        S, pred, parts, est, info = recipe(G_val, N_val, mask, q_tr[qmid][val_idx], b_tr[bmid][val_idx], q_tr[qlast][val_idx])
        yv = y[val_idx]
        report['validation'] = {'rows': int(len(val_idx)), 'groups': info, 'recipe': macro_recall(yv, pred),
                                'components': {k: macro_recall(yv, v.argmax(1)) for k, v in parts.items()}}
        log('validation (held-out training prompts):', json.dumps(report['validation']))
        report['_val_logp'] = (val_idx, G_val)
    # 5. test: frozen models see every usable training row
    S, pred, parts, est, info = recipe(G_test, N_test, usable, q_te[qmid], b_te[bmid], q_te[qlast])
    report['test_groups'] = info; log('test groups:', json.dumps(info))
    np.savez(out_dir / 'parent_scores.npz', ids=test['id'].astype(str).to_numpy(dtype=str), S=S, groups=est, labels=np.asarray(labels))   # for blending with other models later
    sub = pd.DataFrame({'id': test['id'].astype(str), 'model': [labels[i] for i in pred]})
    assert sub.id.is_unique and sub.model.notna().all() and set(sub.model) <= set(labels), 'invalid submission'
    sub.to_parquet(out_dir / 'submission.parquet', index=False)
    report['test_prediction_counts'] = sub.model.value_counts().to_dict(); report['minutes'] = (time.time() - t0) / 60
    (out_dir / 'submission_info.json').write_text(json.dumps({k: v for k, v in report.items() if not k.startswith('_')}, indent=1))
    log(f'wrote {out_dir / "submission.parquet"} ({len(sub)} rows) in {report["minutes"]:.1f} min')
    report['_pred'] = pred; report['_labels'] = labels
    return report

## Install the packages offline and check the stack imports


In [ ]:
wheels = sorted(Path(WHEELS).iterdir())
print(f'installing {len(wheels)} packages: ' + ', '.join(w.name for w in wheels))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q', *map(str, wheels)], capture_output=True, text=True)
print((r.stdout + r.stderr)[-3000:])
if r.returncode: raise SystemExit('The offline install failed: read the pip output above.')
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
check = subprocess.run([sys.executable, 'score_gemma.py', '--check'], capture_output=True, text=True, env=env)
print('\n'.join(l for l in check.stdout.splitlines() if not l.startswith(('🦥', '==((', '   \\', 'O^O', '\\ ', ' "-'))))
if check.returncode: raise SystemExit('Unsloth did not import after the offline install:\n' + check.stderr[-3000:])


## Run the pipeline and write submission.parquet


In [ ]:
import numpy as np, pandas as pd
sys.path.insert(0, '.')
import ntl_pipeline as P

def gemma_fn(texts):
    pd.DataFrame({'text': texts}).to_parquet('/kaggle/working/_texts.parquet')
    r = subprocess.run([sys.executable, 'score_gemma.py', '--base', BASE, '--adapter', ADAPTER, '--head', HEAD, '--texts', '/kaggle/working/_texts.parquet',
                        '--out', '/kaggle/working/_gemma.npz'], capture_output=True, text=True, env=env)
    print('\n'.join(l for l in r.stdout.splitlines() if l.startswith('[')))
    if r.returncode: raise RuntimeError('Gemma scoring failed:\n' + (r.stderr or r.stdout)[-3000:])
    return np.load('/kaggle/working/_gemma.npz')['logp']

report = P.run_pipeline(COMP, POOL, QWEN, BERT, gemma_fn, '/kaggle/working', validate=VALIDATE, val_prompts=VAL_PROMPTS,
                        require_gemma=REQUIRE_GEMMA, use_assignment=USE_ASSIGNMENT, variant=VARIANT, ngram_official_weight=NGRAM_OFFICIAL_WEIGHT)

# proof that the stage-2 adapter and head loaded exactly as trained: score the 64 official rows the stage-2 run saved scores for
chk = pd.read_csv(f'{COMP}/train.csv').text.fillna('').astype(str).head(64).tolist()
got = gemma_fn(chk); saved = np.load(SAVED)['logp']
same = float((got.argmax(1) == saved.argmax(1)).mean())
print(f'Gemma agreement with the stage-2 run on {len(chk)} rows: {same:.1%} (max score difference {np.abs(got - saved).max():.3f})')
if same < 0.95 and REQUIRE_GEMMA:
    raise SystemExit('The Gemma scores do not match the stage-2 run, so the adapter or model setup differs. Do not submit this version.')


## Native expert: score every test passage with the five fold adapters, then blend

In [ ]:
%%writefile native_worker.py
"""Score passages with the public 'native embedding expert' (CHEN Han, Qwen3-Embedding-8B + rank-16 LoRA + 8 pooled depths),
offline, one fold adapter at a time. Needs two T4s: the fp16 backbone is split over both.

  python native_worker.py --base MODELDIR --assets ASSETDIR --query query.csv --out out.npz --work WORKDIR [--folds 0,1,2,3,4]

query.csv has columns id,text. The output holds, for every fold, the plain and centred head logits (centring is applied later,
because it needs prompt groups) and the centred head's bias. A fold's results are saved as soon as it finishes.
"""
import argparse, gc, json, os, time
from pathlib import Path

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F
from safetensors.torch import load_file
from transformers import AutoModel, AutoTokenizer

INSTRUCTION = 'Instruct: Identify the language model that generated the text based on its writing style.\nQuery: '
DEPTHS = (3, 6, 9, 12, 18, 24, 30, 36)
TARGETS = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
MAX_LEN = 512
END_OF_TEXT = 151643
T0 = time.time()


def log(*x): print(f'[{(time.time() - T0) / 60:6.1f}m]', *x, flush=True)


class TaskLoRA(nn.Module):
    """Frozen linear layer plus a rank-16 update kept in fp32 (alpha 32 / rank 16 = scale 2)."""
    def __init__(self, base):
        super().__init__()
        self.base_layer = base
        self.a = nn.Linear(base.in_features, 16, bias=False, device=base.weight.device, dtype=torch.float32)
        self.b = nn.Linear(16, base.out_features, bias=False, device=base.weight.device, dtype=torch.float32)

    def forward(self, x):
        out = self.base_layer(x)
        return (out + self.b(self.a(x.float())) * 2.0).to(out.dtype)


class Expert:
    def __init__(self, base, assets, work):
        torch.set_num_threads(2)
        torch.backends.cudnn.benchmark = False
        for key in ('allow_tf32', 'allow_fp16_reduced_precision_reduction', 'allow_bf16_reduced_precision_reduction'):
            setattr(torch.backends.cuda.matmul, key, False)
        if torch.cuda.device_count() != 2: raise SystemExit(f'the expert needs two GPUs, found {torch.cuda.device_count()}')
        self.assets = Path(assets); base = Path(base)
        self.labels = json.loads((self.assets / 'manifest.json').read_text())['classes']
        index = json.loads((base / 'model.safetensors.index.json').read_text())
        shards = sorted(set(index['weight_map'].values()))
        # configuration and tokenizer come from the asset folder (the training copies); the weight shards are linked from the model input
        stage = Path(work) / 'native-base'; stage.mkdir(parents=True, exist_ok=True)
        for name in ('config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json'):
            src = self.assets / 'base' / name
            if not src.is_file(): raise SystemExit(f'missing {src}')
            (stage / name).write_bytes(src.read_bytes())
        for name in shards:
            if not (base / name).is_file(): raise SystemExit(f'missing weight shard {base / name}')
            link = stage / name
            if not link.exists(): link.symlink_to(base / name)
        cfg = json.loads((stage / 'config.json').read_text())
        if cfg['num_hidden_layers'] != 36 or cfg['hidden_size'] != 4096: raise SystemExit('unexpected backbone configuration')
        self.tok = AutoTokenizer.from_pretrained(stage, local_files_only=True); self.tok.padding_side = 'right'
        if self.tok.pad_token_id is None: self.tok.pad_token = self.tok.eos_token
        placement = {'embed_tokens': 0, 'norm': 1, 'rotary_emb': 1}
        placement.update({f'layers.{i}': 0 if i < 16 else 1 for i in range(36)})
        self.model, info = AutoModel.from_pretrained(stage, local_files_only=True, torch_dtype=torch.float16, device_map=placement,
                                                     attn_implementation='sdpa', output_loading_info=True)
        if any(info.get(k) for k in ('missing_keys', 'mismatched_keys', 'error_msgs', 'unexpected_keys')): raise SystemExit(f'backbone did not load cleanly: {info}')
        self.model.config.use_cache = False
        self.lora = {}
        for name, layer in list(self.model.named_modules()):
            if isinstance(layer, nn.Linear) and name.rsplit('.', 1)[-1] in TARGETS:
                wrapped = TaskLoRA(layer); parent, attr = name.rsplit('.', 1)
                setattr(self.model.get_submodule(parent), attr, wrapped); self.lora[name] = wrapped
        if len(self.lora) != 252: raise SystemExit(f'expected 252 LoRA targets, found {len(self.lora)}')
        self.norms = nn.ModuleList(nn.LayerNorm(4096, device='cuda:0', dtype=torch.float32) for _ in DEPTHS)
        self.plain = nn.Linear(4096 * len(DEPTHS), 18, device='cuda:0', dtype=torch.float32)
        self.centred = nn.Linear(4096 * len(DEPTHS), 18, device='cuda:0', dtype=torch.float32)
        self.model.requires_grad_(False).eval()
        for m in (self.norms, self.plain, self.centred): m.requires_grad_(False).eval()
        self.slots = {}
        for name, m in self.lora.items():
            self.slots[f'encoder.backbone.base_model.model.{name}.lora_A.default.weight'] = m.a.weight
            self.slots[f'encoder.backbone.base_model.model.{name}.lora_B.default.weight'] = m.b.weight
        for prefix, m in (('norms', self.norms), ('plain', self.plain), ('centered', self.centred)):
            self.slots.update({f'{prefix}.{n}': p for n, p in m.named_parameters()})
        if len(self.slots) != 524: raise SystemExit(f'expected 524 trainable tensors, found {len(self.slots)}')
        self.current = None
        log('backbone and LoRA layout ready; GPU memory', [round(torch.cuda.memory_allocated(i) / 2**30, 1) for i in range(2)], 'GB')

    @torch.inference_mode()
    def load_fold(self, fold):
        state = load_file(str(self.assets / f'fold{fold}' / 'trainables.safetensors'))
        if set(state) != set(self.slots): raise SystemExit(f'fold {fold}: adapter tensors do not match: {sorted(set(state) ^ set(self.slots))[:4]}')
        for name, p in self.slots.items():
            t = state[name]
            if t.shape != p.shape or t.dtype != torch.float32 or not torch.isfinite(t).all(): raise SystemExit(f'fold {fold}: bad tensor {name}')
            p.copy_(t.to(p.device))
        del state; gc.collect(); self.current = fold

    def windows(self, texts):
        """Each passage becomes covering windows of at most 512 tokens (the instruction repeated on every later window); no tail is dropped."""
        prefix = self.tok(INSTRUCTION, add_special_tokens=False)['input_ids']
        if self.tok('', add_special_tokens=True)['input_ids'] != [END_OF_TEXT]: raise SystemExit('the tokenizer does not append the end-of-text token')
        capacity = MAX_LEN - len(prefix) - 1; first = MAX_LEN - 1
        full = self.tok([INSTRUCTION + str(t) for t in texts], truncation=False, add_special_tokens=False)['input_ids']
        wins, owner, weight = [], [], []
        for row, toks in enumerate(full):
            wins.append(toks[:first] + [END_OF_TEXT]); owner.append(row); weight.append(max(1, min(len(toks), first) - len(prefix)))
            pos = first
            while pos < len(toks):
                part = toks[pos:pos + capacity]; wins.append(prefix + part + [END_OF_TEXT]); owner.append(row); weight.append(len(part)); pos += len(part)
        return wins, np.asarray(owner), np.asarray(weight, np.float64)

    @torch.inference_mode()
    def run_fold(self, wins, owner, weight, n_texts, batch_size=4):
        order = sorted(range(len(wins)), key=lambda i: (len(wins[i]), wins[i]))
        acc = {k: np.zeros((n_texts, 18)) for k in ('plain', 'centred')}
        for s in range(0, len(order), batch_size):
            pick = order[s:s + batch_size]
            b = self.tok.pad({'input_ids': [wins[i] for i in pick]}, padding=True, return_tensors='pt').to('cuda:0')
            out = self.model(**b, output_hidden_states=True, use_cache=False, return_dict=True)
            if len(out.hidden_states) != 37: raise SystemExit('unexpected number of hidden states')
            last = b['attention_mask'].sum(1) - 1
            parts = []
            for depth, norm in zip(DEPTHS, self.norms):
                h = out.last_hidden_state if depth == 36 else out.hidden_states[depth]
                v = h[torch.arange(len(pick), device=h.device), last.to(h.device)].float().to('cuda:0')
                if depth == 36: v = F.normalize(v, dim=-1)
                parts.append(norm(v))
            feats = torch.cat(parts, -1)
            for key, head in (('plain', self.plain), ('centred', self.centred)):
                np.add.at(acc[key], owner[pick], head(feats).cpu().numpy().astype(np.float64) * weight[pick, None])
            del out, feats, parts, b
            if (s // batch_size) % 100 == 0: log(f'  fold {self.current}: {s}/{len(order)} windows')
        den = np.bincount(owner, weights=weight, minlength=n_texts)[:, None]
        return acc['plain'] / den, acc['centred'] / den


def main():
    ap = argparse.ArgumentParser()
    for name in ('base', 'assets', 'query', 'out', 'work'): ap.add_argument('--' + name, type=Path, required=True)
    ap.add_argument('--folds', default='0,1,2,3,4'); ap.add_argument('--batch-size', type=int, default=4)
    a = ap.parse_args()
    frame = pd.read_csv(a.query, dtype={'id': str}); texts = frame.text.fillna('').astype(str).tolist()
    if not len(texts) or not frame.id.is_unique: raise SystemExit('query must have unique ids and at least one row')
    folds = [int(f) for f in a.folds.split(',')]
    expert = Expert(a.base, a.assets, a.work)
    wins, owner, weight = expert.windows(texts)
    log(f'{len(texts)} passages -> {len(wins)} windows, longest {max(len(w) for w in wins)} tokens, mean {np.mean([len(w) for w in wins]):.0f}')
    res = {'plain': [], 'centred': [], 'bias': []}; timing = []
    for fold in folds:
        t = time.time(); expert.load_fold(fold)
        p, c = expert.run_fold(wins, owner, weight, len(texts), a.batch_size)
        if not (np.isfinite(p).all() and np.isfinite(c).all()): raise SystemExit(f'fold {fold}: non-finite scores')
        res['plain'].append(p); res['centred'].append(c); res['bias'].append(expert.centred.bias.detach().cpu().numpy().copy())
        timing.append({'fold': fold, 'seconds': time.time() - t})
        log(f'fold {fold} done in {(time.time() - t) / 60:.1f} min')
        np.savez(a.out, ids=frame.id.to_numpy(dtype=str), labels=np.asarray(expert.labels), folds=np.asarray(folds[:len(res['plain'])]),
                 plain=np.stack(res['plain']), centred=np.stack(res['centred']), centred_bias=np.stack(res['bias']))
    a.out.with_suffix('.json').write_text(json.dumps({'passages': len(texts), 'windows': len(wins), 'fold_seconds': timing, 'total_seconds': time.time() - T0,
                                                     'peak_gb': [torch.cuda.max_memory_allocated(i) / 2**30 for i in range(2)]}, indent=1))
    log('done')


if __name__ == '__main__': main()

In [ ]:
%%writefile native_fusion.py
"""Combine the native expert's raw head logits (native_worker.py output) with our parent scores.

native_logp(): per fold, calibrate the plain head and the prompt-centred head with the author's stored scales, centre the second inside the
estimated prompt group (groups of 2-18 passages only; other rows keep the plain head), mix them at the fold's stored weight, then average the
five folds' log-probabilities.
blend_parent_native(): z-scored weighted sum, the same way the rest of the pipeline blends.
"""
import json
from pathlib import Path
import numpy as np
from scipy.special import log_softmax


def native_logp(npz_path, assets, groups, labels):
    z = np.load(npz_path, allow_pickle=False)
    saved = z['labels'].tolist()
    if sorted(saved) != sorted(labels): raise ValueError('class names differ between the expert and the pipeline')
    order = [saved.index(l) for l in labels]
    plain, cent, bias, folds = z['plain'][:, :, order], z['centred'][:, :, order], z['centred_bias'][:, order], z['folds'].tolist()
    groups = np.asarray(groups)
    if plain.shape[1] != len(groups): raise ValueError('group labels do not match the scored passages')
    inv = np.unique(groups, return_inverse=True)[1]; count = np.bincount(inv); eligible = (count[inv] >= 2) & (count[inv] <= 18)
    out = []
    for i, fold in enumerate(folds):
        cal = json.loads((Path(assets) / 'calibration' / f'fold{fold}.json').read_text())['models']['qwen8']
        if not cal['centered_head_trained'] or cal['outer_labels_used']: raise ValueError(f'fold {fold}: unexpected calibration record')
        mean = np.zeros((len(count), cent.shape[2])); np.add.at(mean, inv, cent[i]); mean /= count[:, None]
        a = log_softmax(plain[i] * cal['plain_scale'], axis=1)
        b = log_softmax((cent[i] - mean[inv] + bias[i]) * cal['centered_scale'], axis=1)
        w = np.where(eligible, cal['selected_centered_weight'], 0.0)[:, None]
        out.append(log_softmax((1 - w) * a + w * b, axis=1))
    return np.mean(out, axis=0)


def blend_parent_native(parent, native, native_weight):
    z = lambda a: a / a.std()
    return (1 - native_weight) * z(parent) + native_weight * z(native)

In [ ]:
import shutil
from native_fusion import native_logp, blend_parent_native
shutil.copy('/kaggle/working/submission.parquet', '/kaggle/working/submission_parent.parquet')
parent = np.load('/kaggle/working/parent_scores.npz', allow_pickle=False)
ids, labels, S_parent, groups = parent['ids'].tolist(), parent['labels'].tolist(), parent['S'], parent['groups']
test = pd.read_csv(f'{COMP}/test.csv'); test['text'] = test.text.fillna('').astype(str)
assert len(test) == len(ids) and test.id.astype(str).tolist() == ids, 'test rows differ from the ones the parent pipeline scored'
pd.DataFrame({'id': ids, 'text': test.text.values}).to_csv('/kaggle/working/native_query.csv', index=False)
Path('/kaggle/temp/work').mkdir(parents=True, exist_ok=True)
t = time.time()
p = subprocess.Popen([sys.executable, '-u', 'native_worker.py', '--base', NATIVE_BASE, '--assets', ASSETS, '--query', '/kaggle/working/native_query.csv',
                      '--out', '/kaggle/working/native_scores.npz', '--work', '/kaggle/temp/work', '--folds', '0,1,2,3,4'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, TRANSFORMERS_OFFLINE='1', HF_HUB_OFFLINE='1'))
tail = []
for line in p.stdout:
    print(line, end=''); tail = (tail + [line])[-40:]
if p.wait():
    if REQUIRE_NATIVE: raise SystemExit('The native expert failed:\n' + ''.join(tail[-25:]))
    print('!!! native expert failed, keeping the parent blend')
else:
    print(f'native expert wall time {(time.time() - t) / 60:.1f} min')
    N = native_logp('/kaggle/working/native_scores.npz', ASSETS, groups, labels)
    final = blend_parent_native(S_parent, N, NATIVE_WEIGHT)
    pred = final.argmax(1)
    sub = pd.DataFrame({'id': ids, 'model': [labels[i] for i in pred]})
    assert sub.id.is_unique and sub.model.notna().all() and set(sub.model) <= set(labels), 'invalid submission'
    sub.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('agreement  parent vs final %.1f%%   native vs final %.1f%%   parent vs native %.1f%%' % (
        100 * (S_parent.argmax(1) == pred).mean(), 100 * (N.argmax(1) == pred).mean(), 100 * (S_parent.argmax(1) == N.argmax(1)).mean()))
    print('prediction counts:', sub.model.value_counts().to_dict())
    info = json.loads(Path('/kaggle/working/submission_info.json').read_text())
    info['native'] = {'weight': NATIVE_WEIGHT, 'minutes': (time.time() - t) / 60, 'prediction_counts': sub.model.value_counts().to_dict()}
    Path('/kaggle/working/submission_info.json').write_text(json.dumps(info, indent=1))
    print(f'wrote /kaggle/working/submission.parquet ({len(sub)} rows)')

## L24 expert: score every test passage with the five q8v5 fold adapters, then the final blend

In [ ]:
%%writefile l24_worker.py
"""Score passages with the public 'L24' LoRA experts (dhruvtibarewal/nlm-l24-adapters-v5: q14v5, q8v5, ll8v5), offline.

Recipe (read from the public notebooks, own implementation): the base model is TRUNCATED to its first `keep` decoder layers, a rank-16 LoRA
on every attention/MLP projection is applied, the hidden states at `layers` (indices into hidden_states, 0 = embeddings) are mean-pooled over
real tokens (first token masked), each goes through its own LayerNorm, and the concatenation feeds two linear heads: plain and prompt-centred.

  python l24_worker.py --base DIR --adapters A0,A1,... --query q.csv --out out.npz [--keep 24] [--layers 2,4,6,8,12,16,20,24]

query.csv has columns id,text. For every adapter the output holds the raw plain logits and the raw centred-head logits (centring needs prompt
groups, so it is applied by the caller: centred = raw - group_mean(raw) + bias). LoRA is applied as a separate fp32 low-rank path (no merging, so
no rounding drift between adapters). Results are written after every adapter.
"""
import argparse, gc, json, re, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer

T0 = time.time()
TARGETS = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')


def log(*x): print(f'[{(time.time() - T0) / 60:6.1f}m]', *x, flush=True)


class LoRALinear(nn.Module):
    """Frozen linear layer plus an optional fp32 low-rank update. The factors live in a plain tuple (not registered), so model.to(dtype) never touches them."""
    def __init__(self, base):
        super().__init__(); self.base_layer = base; self.ab = None; self.scale = 1.0

    def forward(self, x):
        out = self.base_layer(x)
        if self.ab is None: return out
        a, b = self.ab
        return (out + (x.float() @ a.t() @ b.t()) * self.scale).to(out.dtype)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--base', required=True); ap.add_argument('--adapters', required=True); ap.add_argument('--query', required=True); ap.add_argument('--out', required=True)
    ap.add_argument('--keep', type=int, default=24); ap.add_argument('--layers', default='2,4,6,8,12,16,20,24')
    ap.add_argument('--max-len', type=int, default=384); ap.add_argument('--bs', type=int, default=16)
    a = ap.parse_args()
    adapters = a.adapters.split(','); layers = [int(x) for x in a.layers.split(',')]
    frame = pd.read_csv(a.query, dtype={'id': str}); texts = frame.text.fillna('').astype(str).tolist()
    if not len(texts) or not frame.id.is_unique: raise SystemExit('query must have unique ids and at least one row')

    tok = AutoTokenizer.from_pretrained(a.base)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(a.base); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    full_layers = tc.num_hidden_layers; tc.num_hidden_layers = a.keep; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None: tc.layer_types = tc.layer_types[:a.keep]
    model = AutoModel.from_pretrained(a.base, config=cfg, dtype=torch.float16, device_map='auto', max_memory={i: '13GiB' for i in range(torch.cuda.device_count())}).eval()
    first = next(model.parameters()).device
    wrappers = {}
    for name, mod in list(model.named_modules()):
        if isinstance(mod, nn.Linear) and name.rsplit('.', 1)[-1] in TARGETS and re.match(r'layers\.\d+\.', name):
            parent, attr = name.rsplit('.', 1); w = LoRALinear(mod); setattr(model.get_submodule(parent), attr, w); wrappers[name] = w
    if len(wrappers) != 7 * a.keep: raise SystemExit(f'expected {7 * a.keep} LoRA targets, found {len(wrappers)}')
    log(f'base loaded: kept {a.keep} of {full_layers} layers, {len(wrappers)} LoRA targets, devices {sorted({str(w.base_layer.weight.device) for w in wrappers.values()})}')

    enc = tok(texts, truncation=True, max_length=a.max_len)['input_ids']
    order = np.argsort([len(x) for x in enc], kind='stable'); N = len(enc)
    log(f'{N} passages, mean {np.mean([len(x) for x in enc]):.0f} tokens, longest {max(len(x) for x in enc)}')

    @torch.no_grad()
    def forward_batch(b):
        Lm = max(len(enc[j]) for j in b)
        ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
        for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
        ids, att = ids.to(first), att.to(first)
        hs = model(input_ids=ids, attention_mask=att, output_hidden_states=True, use_cache=False).hidden_states
        redo = False
        if not all(bool(torch.isfinite(hs[l]).all()) for l in layers):   # fp16 overflow in un-normed deep states: redo this batch in bf16
            del hs; model.to(torch.bfloat16); redo = True
            hs = model(input_ids=ids, attention_mask=att, output_hidden_states=True, use_cache=False).hidden_states
        mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
        pooled = [((hs[l].float() * mask.to(hs[l].device)).sum(1) / mask.sum(1).to(hs[l].device)).to(first) for l in layers]
        del hs
        if redo: model.to(torch.float16)
        return pooled, redo

    res = {'plain': [], 'centred_raw': [], 'centred_bias': []}; timing = []; overflow = 0
    for ai, ad in enumerate(adapters):
        t = time.time(); ad = Path(ad)
        cfg_a = json.loads((ad / 'adapter_config.json').read_text()); scale = cfg_a['lora_alpha'] / cfg_a['r']
        sd = load_file(str(ad / 'adapter_model.safetensors')); used = skipped = 0
        for w in wrappers.values(): w.ab = None
        for k, A in sd.items():
            if '.lora_A.' not in k: continue
            m = re.search(r'layers\.(\d+)\.((?:self_attn|mlp)\.\w+)\.lora_A\.', k)
            if not m: raise SystemExit(f'unexpected adapter key {k}')
            if int(m.group(1)) >= a.keep: skipped += 1; continue
            w = wrappers[f'layers.{m.group(1)}.{m.group(2)}']; B = sd[k.replace('.lora_A.', '.lora_B.')]
            dev = w.base_layer.weight.device; w.ab = (A.float().to(dev), B.float().to(dev)); w.scale = scale; used += 1
        if used != len(wrappers): raise SystemExit(f'{ad.name}: adapter covers {used} of {len(wrappers)} targets ({skipped} skipped beyond layer {a.keep})')
        H = torch.load(ad / 'heads.pt', map_location='cpu', weights_only=True)
        hid = H['norms.0.weight'].shape[0]; norms = []
        for i in range(len(layers)):
            n = nn.LayerNorm(hid); n.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']}); norms.append(n.to(first).float())
        Wp, bp = H['plain.weight'].float().to(first), H['plain.bias'].float().to(first)
        Wc, bc = H['centered.weight'].float().to(first), H['centered.bias'].float().to(first)
        plain = np.zeros((N, 18), np.float32); cent = np.zeros((N, 18), np.float32)
        for s in range(0, N, a.bs):
            b = order[s:s + a.bs]; pooled, redo = forward_batch(b); overflow += int(redo)
            with torch.no_grad():
                Z = torch.cat([n(p) for n, p in zip(norms, pooled)], -1)
                plain[b] = (Z @ Wp.t() + bp).cpu().numpy(); cent[b] = (Z @ Wc.t() + bc).cpu().numpy()
            if (s // a.bs) % 40 == 0: log(f'  {ad.name}: {s}/{N}')
        if not (np.isfinite(plain).all() and np.isfinite(cent).all()): raise SystemExit(f'{ad.name}: non-finite scores')
        res['plain'].append(plain); res['centred_raw'].append(cent); res['centred_bias'].append(bc.cpu().numpy())
        timing.append({'adapter': ad.name, 'seconds': time.time() - t}); del sd, H; gc.collect()
        log(f'{ad.name} done in {(time.time() - t) / 60:.1f} min (bf16 redo batches so far: {overflow})')
        np.savez(a.out, ids=frame.id.to_numpy(dtype=str), adapters=np.asarray([Path(x).name for x in adapters[:ai + 1]]),
                 plain=np.stack(res['plain']), centred_raw=np.stack(res['centred_raw']), centred_bias=np.stack(res['centred_bias']))
    Path(a.out).with_suffix('.json').write_text(json.dumps({'passages': N, 'keep': a.keep, 'layers': layers, 'adapter_seconds': timing, 'bf16_redo_batches': overflow,
                                                           'total_seconds': time.time() - T0, 'peak_gb': [torch.cuda.max_memory_allocated(i) / 2**30 for i in range(torch.cuda.device_count())]}, indent=1))
    log('done')


if __name__ == '__main__': main()

In [ ]:
%%writefile l24_fusion.py
"""Turn l24_worker.py output into log-probabilities. For rows inside an estimated prompt group of 2-18 passages use the CENTRED head
(raw logits - group mean + the head's bias); other rows use the plain head. Average the log-probs over the fold adapters.
Out-of-fold (experiments/l24_blend_local.py): centred head alone beat plain (0.665 vs 0.615) and beat any plain/centred mix when blended with the native expert."""
import numpy as np
from scipy.special import log_softmax


def l24_logp(npz_path, groups, min_group=2, max_group=18):
    z = np.load(npz_path, allow_pickle=False)
    plain, craw, cbias = z['plain'].astype(np.float64), z['centred_raw'].astype(np.float64), z['centred_bias'].astype(np.float64)
    groups = np.asarray(groups)
    if plain.shape[1] != len(groups): raise ValueError('group labels do not match the scored passages')
    inv = np.unique(groups, return_inverse=True)[1]; count = np.bincount(inv); eligible = ((count[inv] >= min_group) & (count[inv] <= max_group))[:, None]
    out = []
    for a in range(plain.shape[0]):
        mean = np.zeros((len(count), plain.shape[2])); np.add.at(mean, inv, craw[a]); mean /= count[:, None]
        out.append(log_softmax(np.where(eligible, craw[a] - mean[inv] + cbias[a], plain[a]), axis=1))
    return log_softmax(np.mean(out, axis=0), axis=1)

In [ ]:
from l24_fusion import l24_logp
shutil.copy('/kaggle/working/submission.parquet', '/kaggle/working/submission_two_way.parquet')   # parent + native, written by the cell above
experts = [dict(name='q8v5', base=Q8_BASE, keep=24, bs=16, adapters=[f'{L24_ADAPTERS}/q8v5_f{k}' for k in range(5)])]
E = []
for e in experts:
    t = time.time(); out = f'/kaggle/working/l24_{e["name"]}.npz'
    p = subprocess.Popen([sys.executable, '-u', 'l24_worker.py', '--base', e['base'], '--adapters', ','.join(e['adapters']), '--query', '/kaggle/working/native_query.csv',
                          '--out', out, '--keep', str(e['keep']), '--bs', str(e['bs'])], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, TRANSFORMERS_OFFLINE='1', HF_HUB_OFFLINE='1'))
    tail = []
    for line in p.stdout:
        print(line, end=''); tail = (tail + [line])[-40:]
    if p.wait():
        if REQUIRE_L24: raise SystemExit(f'The L24 expert {e["name"]} failed:\n' + ''.join(tail[-25:]))
        print(f'!!! L24 expert {e["name"]} failed, leaving it out'); continue
    E.append(l24_logp(out, groups)); print(f'{e["name"]} wall time {(time.time() - t) / 60:.1f} min')
if E:
    z = lambda a: a / a.std()
    final = W_PARENT * z(S_parent) + W_NATIVE * z(N) + W_L24 * np.mean([z(x) for x in E], axis=0)
    pred = final.argmax(1)
    sub = pd.DataFrame({'id': ids, 'model': [labels[i] for i in pred]})
    assert sub.id.is_unique and sub.model.notna().all() and set(sub.model) <= set(labels), 'invalid submission'
    sub.to_parquet('/kaggle/working/submission.parquet', index=False)
    two = pd.read_parquet('/kaggle/working/submission_two_way.parquet')
    print('agreement  final vs two-way %.1f%%   final vs L24 alone %.1f%%   native vs L24 %.1f%%' % (100 * (two.model.values == sub.model.values).mean(),
          100 * (E[0].argmax(1) == pred).mean(), 100 * (N.argmax(1) == E[0].argmax(1)).mean()))
    print('prediction counts:', sub.model.value_counts().to_dict())
    info = json.loads(Path('/kaggle/working/submission_info.json').read_text())
    info['l24'] = {'weights': [W_PARENT, W_NATIVE, W_L24], 'experts': [e['name'] for e in experts], 'prediction_counts': sub.model.value_counts().to_dict()}
    Path('/kaggle/working/submission_info.json').write_text(json.dumps(info, indent=1))
    print(f'wrote /kaggle/working/submission.parquet ({len(sub)} rows)')

## Result


In [ ]:
print(Path('/kaggle/working/submission_info.json').read_text())
sub = pd.read_parquet('/kaggle/working/submission.parquet'); print(sub.shape, sub.dtypes.to_dict()); print(sub.head())
